# 🎓 초보 학습자를 위한 과학 개념 쉬운 설명 LLM 파인튜닝

Google Colab 무료 **T4 GPU**에서 **Unsloth**를 사용하여 소형 LLM을 파인튜닝합니다.
본 프로젝트는 기말 프로젝트의 **10번 시나리오: 도메인 용어 설명가**를 기반으로 진행합니다.

* **목적:** 과학 개념을 초보 학습자가 이해하기 쉬운 짧은 문장과 간단한 예시로 설명하는 LLM 만들기
* **선택 시나리오:** 10. 도메인 용어 설명가
* **프로젝트명:** 초보 학습자를 위한 과학 개념 쉬운 설명 LLM
* **기본 모델:** `unsloth/Qwen2.5-0.5B-Instruct`
* **데이터셋:** `science_en`
* **방식:** LoRA / QLoRA 4bit 파인튜닝




## 0. GPU 확인


In [2]:
!nvidia-smi


Sun Jun 14 11:27:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Unsloth 설치
코랩 최신 환경에 맞춰 자동 설치됩니다. (1~2분 소요)


In [3]:
%%capture
import os
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets


## 2. 학습 데이터 — HuggingFace Hub에서 받기
손으로 만든 예시가 아니라 **실제 한국어 교육 데이터셋**을 HF Hub에서 받아 20시나리오 포맷으로 변환합니다.

추천 데이터셋(소스별 N개만 streaming 추출 → 대형도 전체 다운로드 없음):
- `JosephLee/korean-socratic-qa` (105k) — 소크라테스 문답
- `kuotient/orca-math-word-problems-193k-korean` (193k) — 수학 단계 풀이
- `beomi/KoAlpaca-v1.1a` (21k) — 일반 instruction 베이스
- `jojo0217/korean_safe_conversation` (27k, Apache-2.0) — 정서지원·공감
- `neuralfoundry-coder/aihub-korean-education-instruct-sample` (6k) — 교육 상담·분석


In [4]:
# 저장소 clone (스크립트/레지스트리 사용)
REPO = 'https://github.com/xide-projext/edu-llm-colab-unsloth.git'
import os
if not os.path.exists('edu-llm-colab-unsloth'):
    !git clone -q $REPO
%cd edu-llm-colab-unsloth

# HF에서 소스별 8000개씩 받아 data/hf_train.jsonl 생성 (≈5분)
!python scripts/fetch_hf_datasets.py --per-source 8000 --val-ratio 0.05 --only science_en
from pathlib import Path
DATA_PATH = "data/hf_train.jsonl"
assert Path(DATA_PATH).exists(), "fetch 실패 — 위 셀 로그 확인"
print("학습 데이터 준비 완료:", DATA_PATH)


/content/edu-llm-colab-unsloth

▶ [science_en] allenai/sciq (목표 8000개, license=CC-BY-NC-3.0)
README.md: 100% 7.02k/7.02k [00:00<00:00, 14.9MB/s]
  ✅ 8000개 수집

총 8000개 수집 → 분할 저장
  저장: data/hf_train.jsonl (7600개)
  저장: data/hf_val.jsonl (400개)

[소스별 분포]
  allenai/sciq                                            8000

✅ 완료 — 학습엔 data/hf_train.jsonl 를 사용하세요 (seed_train.jsonl 은 예시 참고용)
학습 데이터 준비 완료: data/hf_train.jsonl


## 3. 모델 로드 (4bit)


In [5]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024
MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,            # 자동 감지 (T4=float16)
    load_in_4bit = True,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


## 4. LoRA 어댑터 추가
전체 가중치가 아닌 일부 행렬만 학습해 메모리/시간을 절약합니다.


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.7 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


## 5. 데이터 포맷팅 (채팅 템플릿)
`instruction`(시나리오 지시) → system, `input` → user, `output` → assistant 로 매핑합니다.


In [7]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def to_text(ex):
    msgs = [
        {"role": "system",    "content": ex["instruction"]},
        {"role": "user",      "content": ex["input"]},
        {"role": "assistant", "content": ex["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

dataset = load_dataset("json", data_files=DATA_PATH, split="train")
dataset = dataset.map(to_text)
print("샘플 수:", len(dataset))
print(dataset[0]["text"][:600])


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

샘플 수: 7600
<|im_start|>system
시나리오[10]: 과학 개념을 근거와 함께 설명하고 정답을 제시하세요.<|im_end|>
<|im_start|>user
What do covalent bonds give atoms a more stable arrangement of?<|im_end|>
<|im_start|>assistant
Covalent bonds form because they give atoms a more stable arrangement of electrons. Look at the oxygen atoms in the Figure above . Alone, each oxygen atom has six valence electrons. By sharing two pairs of valence electrons, each oxygen atom has a total of eight valence electrons. This fills its outer energy level, giving it the most stable arrangement of electrons. The shared electrons are attracted to both oxygen


## 6. 트레이너 설정 & 학습
데모용 `max_steps=60`. 실제 학습에선 `num_train_epochs`로 바꾸세요.


In [8]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,

        learning_rate = 2e-4,
        num_train_epochs = 1,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer_stats = trainer.train()


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/7600 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,600 | Num Epochs = 1 | Total steps = 950
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,3.318805
2,3.310333
3,3.087082
4,2.635908
5,2.641465
6,2.498774
7,2.279871
8,2.090541
9,1.883616
10,1.910738


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-950/tokenizer_config.json.


## 7. 추론 테스트
학습한 시나리오 톤이 나오는지 즉석에서 확인합니다.


In [ ]:
import torch

FastLanguageModel.for_inference(model)

system_prompt = """
시나리오[10]: 도메인 용어 설명가.
과학 개념을 초보 학습자가 이해하기 쉬운 짧은 문장으로 설명하세요.
가능하면 간단한 일상 예시를 함께 제시하세요.
"""

test_questions = [
    "What is gravity?",
    "What is light?",
    "What is heat?"
]

for question in test_questions:
    print("=" * 60)
    print("Question:", question)
    print("Answer:")

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    attention_mask = torch.ones_like(input_ids).to("cuda")

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=80,
        temperature=0.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    generated_tokens = outputs[0][input_ids.shape[-1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    print(answer.strip())
    print("\n")

Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is gravity?
Answer:


Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Gravity is the force that attracts all objects with mass toward each other. It acts between any two masses, but it does not act directly on a single object. Gravity can be described by Newton’s law of universal gravitation.

Answer: the force that attracts all objects with mass toward each other


Question: What is light?
Answer:


Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Light is a form of energy. It travels through space and can be reflected, refracted, or absorbed by matter.

Answer: a form of energy


Question: What is heat?
Answer:
Heat is the energy of motion. It can be transferred from one place to another by a transfer of energy.

Answer: energy of motion




In [10]:
import torch
import pandas as pd

system_prompt = """
Scenario 10: You are a science concept explainer for beginner learners.
Explain the science concept in 1-2 simple English sentences.
If possible, include a simple everyday example.
"""

test_questions = [
    "What is gravity?",
    "What is light?",
    "What is heat?"
]

def generate_answer(question):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    attention_mask = torch.ones_like(input_ids).to("cuda")

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=80,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    generated_tokens = outputs[0][input_ids.shape[-1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return answer.strip()

results = []

# Base Model: LoRA adapter 비활성화
with model.disable_adapter():
    for q in test_questions:
        base_answer = generate_answer(q)
        results.append({
            "Question": q,
            "Base Model": base_answer,
            "Fine-tuned Model": ""
        })

# Fine-tuned Model: LoRA adapter 활성화
for i, q in enumerate(test_questions):
    fine_tuned_answer = generate_answer(q)
    results[i]["Fine-tuned Model"] = fine_tuned_answer

df = pd.DataFrame(results)
pd.set_option("display.max_colwidth", None)
display(df)

Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=80

,Question,Base Model,Fine-tuned Model
0,What is gravity?,Gravity is a natural force between all objects that attracts them towards each other. Imagine you're holding a ball with your hand; the ball feels heavier than it does when you release it. That's because of gravity! It's what keeps planets and stars orbiting around us.,Gravity is the force that pulls objects toward each other. It’s what keeps Earth in orbit around the Sun.\n\nAnswer: the force that pulls objects toward each other
1,What is light?,"Light is a form of energy that travels through space and interacts with matter. It's what we see as colors when we look at objects, like how different shades of red appear to us. Light can be seen from all directions, including inside our bodies.",Light is energy that travels through space. Light can be seen as a form of electromagnetic radiation.\n\nAnswer: energy that travels through space
2,What is heat?,"Heat is a measure of energy that transfers from one object to another through contact. It's like when you feel warm when you touch a hot pan or a cup of coffee on your hand. Heat can be caused by various processes like burning, melting, or radiating sunlight.","Heat is energy that moves from one place to another. Heat can be transferred by conduction , convection , or radiation . Conduction transfers heat through direct contact between two objects. Convection transfers heat through moving air. Radiation transfers heat through electromagnetic waves.\n\nAnswer: energy that moves from one place to another"


## 8. 저장 & 내보내기
- **LoRA 어댑터**: 가볍게 재사용 (수십 MB)
- **GGUF**: Ollama / LM Studio / llama.cpp 용 (옵션, 시간 소요)


In [11]:
# (1) LoRA 어댑터만 저장
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("LoRA 저장 완료 → lora_model/")

# (2) GGUF 내보내기 (필요할 때 주석 해제)
# model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")

# (3) 구글 드라이브에 백업 (필요할 때 주석 해제)
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r lora_model /content/drive/MyDrive/


Unsloth: Restored added_tokens_decoder metadata in lora_model/tokenizer_config.json.


LoRA 저장 완료 → lora_model/


In [12]:
# GGUF 형식으로 내보내기
model.save_pretrained_gguf(
    "model_gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)

print("GGUF export 완료 → model_gguf/")

# 생성된 GGUF 파일 확인
!find model_gguf -type f -name "*.gguf" -print
!ls -lh model_gguf

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in model_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:09<00:00,  9.75s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:07<00:00,  7.04s/it]


Unsloth: Merge process complete. Saved to `/content/edu-llm-colab-unsloth/model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9631 (llama-b9631-bin-ubuntu-x64.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['model_gguf_gguf/qwen2.5-0.5b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfu

---
### 다음 단계
1. 데이터를 더 받으려면 `--per-source` 를 키우세요 (예: 20000). 소스 추가는 `scripts/fetch_hf_datasets.py` 의 SOURCES 레지스트리에서.
2. 특정 시나리오만 학습하려면 `--only socratic math` 처럼 지정.
3. 시나리오별 성능은 `data/scenarios.json` 의 *기대치(expectation)* 기준으로 평가하세요.
4. `data/seed_train.jsonl` 은 학습용이 아니라 시나리오별 **목표 톤 예시(참고용)** 입니다.
